In [ ]:
# ==============================================================================
# 📓 Enterprise RAG Cookbook: 07_evaluation_and_metrics.ipynb
# ==============================================================================
# الهدف: تطبيق آلية تقييم نظم الـ RAG في بيئات الإنتاج (Enterprise RAG Evaluation):
# 1. LLM-as-a-Judge Pattern
# 2. Evaluation of RAG Triad (Context Relevance, Groundedness, Answer Relevance)
# 3. Pydantic Evaluation Schemas & Automated Scoring Pipeline
# ==============================================================================

# !pip install langchain-core langchain-openai pydantic

import os
from typing import List
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

print("✅ تم استيراد مكتبات التقييم والـ Metrics بنجاح!")

<div dir="rtl">

## 1. إطار عمل التقييم: RAG Triad Metrics & Pydantic Schemas

في هذا الجزء، نبني **النموذج الهيكلي (Pydantic Schema)** للتقييم، حيث يلتزم الـ LLM Judge بإرجاع نتيجة رقمية من `0.0` إلى `1.0` لكل معيار من المعايير الثلاثة، مع ذكر السبب والتحليل (Reasoning).

</div>

In [ ]:
# ==============================================================================
# 1. Pydantic Evaluation Schemas for RAG Triad
# ==============================================================================

class RAGTriadMetric(BaseModel):
    score: float = Field(
        description="Score between 0.0 (Worst) and 1.0 (Best)."
    )
    reasoning: str = Field(
        description="Detailed step-by-step reasoning explaining why this score was assigned."
    )

class FullRAGEvaluation(BaseModel):
    context_relevance: RAGTriadMetric = Field(
        description="Measures if retrieved chunks are relevant to the query."
    )
    groundedness: RAGTriadMetric = Field(
        description="Measures if the answer is strictly based ONLY on retrieved context (No Hallucination)."
    )
    answer_relevance: RAGTriadMetric = Field(
        description="Measures if the answer directly addresses the original query."
    )
    overall_score: float = Field(
        description="Weighted aggregate average score across all three metrics."
    )

print("✅ تم تصميم الـ Pydantic Evaluation Schemas للإنتاج!")

<div dir="rtl">

## 2. بناء الـ LLM-as-a-Judge Prompt

السر في نظام التقييم الدقيق هو الـ **System Prompt** المحكم المحايد الذي يعمل كقاضٍ يفحص الثلاثية بالخطوات.

</div>

In [ ]:
# ==============================================================================
# 2. Enterprise LLM-as-a-Judge Prompt
# ==============================================================================

EVALUATOR_SYSTEM_PROMPT = """You are an expert AI Auditor evaluating an Enterprise RAG Pipeline.
Your job is to objectively score the RAG Triad based strictly on the provided inputs.

INPUTS TO EVALUATE:
1. Query: The user's original question.
2. Context: The retrieved context chunks provided to the LLM.
3. Answer: The final generated answer produced by the system.

SCORING GUIDELINES:
- Context Relevance: Does Context contain the information needed to answer Query? (0.0 = Irrelevant, 1.0 = Highly relevant).
- Groundedness: Is Answer completely supported by Context alone? Any claim not in Context lowers the score (0.0 = Pure Hallucination, 1.0 = 100% Grounded).
- Answer Relevance: Does Answer directly answer Query without extra unnecessary chatter? (0.0 = Off-topic, 1.0 = Perfectly direct answer).
"""

eval_prompt = ChatPromptTemplate.from_messages([
    ("system", EVALUATOR_SYSTEM_PROMPT),
    ("human", "QUERY: {query}\n\nCONTEXT: {context}\n\nGENERATED ANSWER: {answer}")
])

# إعداد القاضي (LLM Judge)
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_evaluator = judge_llm.with_structured_output(FullRAGEvaluation)

eval_chain = eval_prompt | structured_evaluator
print("✅ تم تجهيز سلاسل التقييم الآلي (Evaluation Chain)!")

<div dir="rtl">

## 3. محاكاة تشغيل وتتبع نتائج التقييم (Automated Evaluation Pipeline)

نقوم الآن بتمرير عينتين لنرى كيف يستكشف القاضي الـ **Hallucination** والـ **Context Irrelevance** تلقائياً.

</div>

In [ ]:
# ==============================================================================
# 3. Automated Evaluation Function & Test Cases
# ==============================================================================

def evaluate_rag_sample(query: str, context: str, answer: str) -> FullRAGEvaluation:
    """دالة التقييم الآلية التي ترجع تقريراً شاملاً بالدرجات والتحليل"""
    return eval_chain.invoke({
        "query": query,
        "context": context,
        "answer": answer
    })

# عينة اختبار بها هبد (Hallucination Test Case)
test_query = "What is the annual leave allowance?"
test_context = "[Doc 1] Probation period lasts 6 months. Annual leave grants 21 business days after probation."
bad_answer = "You get 21 business days of annual leave. Also, the company pays a $500 travel allowance every year."

print(f"❓ Query: {test_query}")
print(f"📄 Context: {test_context}")
print(f"🤖 Answer: {bad_answer}\n")
print("⚙️ [جاهز للاستدعا الفعلي عند إدخال الـ API Key لتوليد التقرير الآلي]")

<div dir="rtl">

## 📝 ملخص معايير التقييم في الشركات (Production Evaluation Best Practices)

1. **CI/CD Pipeline Integration:** تشغيل سكريبت التقييم بصفة دورية على Test Dataset مكونة من ~100 سؤال وجواب مرجعي مع كل تعديل في الـ Pipeline.
2. **استخدام نماذج أقوى كقاضٍ:** يُفضل استخدام نموذج مثل `gpt-4o` لتقييم مخرجات النماذج الأصغر مثل `gpt-4o-mini` لضمان حيادية ودقة القضاء.
3. **التنبيه الآلي (Alerts):** إن حاجز الـ Groundedness Score إذا هبط عن `0.85` في بيئة الإنتاج يُعتبر مؤشراً خطيراً لاستدعاء مراجعة النظام الفورية.

</div>